# costsight — End-to-end walkthrough

Project 13 · Cloud Computing · Spring 2025–2026

This notebook reproduces every Phase 1 + Phase 2 deliverable in one go. Run all cells top-to-bottom; the dashboard tabs you saw on screen all derive from these same APIs.

Sections:
1. Generate a synthetic AWS-CUR-shaped dataset (with scenario presets).
2. Run all four detectors (Z-Score, STL, Isolation Forest, Ensemble).
3. Build severity-banded alerts and root-cause attribution.
4. Evaluate Precision / Recall / F1 + bootstrap confidence intervals + Wilcoxon paired test.
5. Forecast the next 14 days and project a monthly bill.
6. Cluster alerts into incidents.
7. Time-to-detect and cost-saved estimates.
8. Real AWS CUR ingestion sanity check.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parents[0] if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np

pd.options.display.max_rows = 12
pd.options.display.float_format = '{:,.2f}'.format

## 1. Synthetic CUR generator

`scenario` controls which anomaly mix gets injected. `default` matches the 25-seed benchmark. Switch to `drift_heavy`, `spike_storm`, `stealth_leak`, `multi_region`, `weekend_camouflage`, or `calm` for different shapes.

In [ ]:
from cloud_anomaly.synthetic_data import generate, SCENARIOS

for name, blurb in SCENARIOS.items():
    print(f'  {name:<22} {blurb}')

cur_df, labels_df, anomalies = generate(n_days=90, seed=42, scenario='default')
print(f'\nRows: {len(cur_df):,}  ·  services: {cur_df["service"].nunique()}  ·  ground-truth positives: {labels_df["is_anomaly"].sum()}')
cur_df.head()

## 2. Run all four detectors

Every detector exposes the same `detect(long_df) -> DataFrame` signature so the rest of the pipeline is detector-agnostic.

In [ ]:
from cloud_anomaly.preprocessing import aggregate_by_service
from cloud_anomaly.detectors import DETECTORS

long = aggregate_by_service(cur_df)
detections = {name: fn(long) for name, fn in DETECTORS.items()}

summary = pd.DataFrame({
    name: {'flagged': int(d['is_anomaly'].sum()), 'rate %': 100 * d['is_anomaly'].mean()}
    for name, d in detections.items()
}).T
summary

## 3. Severity-banded alerts + root-cause attribution

Severity = deviation × duration × dollar_impact, clipped to [0, 1] and bucketed into LOW/MEDIUM/HIGH. Attribution decomposes the day's spend along (region, usage_type, tag_team, tag_environment) and reports the dimension that drove the increase.

In [ ]:
from cloud_anomaly.alerts import build_alerts
from cloud_anomaly.attribution import attribute

alerts_stl = build_alerts(detections['stl'], detector_name='stl', dataset_days=long['date'].nunique())
attribution = attribute(cur_df, alerts_stl)
alerts_stl[['date', 'service', 'cost', 'severity', 'severity_score']].head()

In [ ]:
attribution[['date', 'service', 'severity', 'summary']].head()

## 4. Evaluation

Per-anomaly-type Precision/Recall/F1 + bootstrap CI + Wilcoxon paired test on the per-seed benchmark output.

In [ ]:
from cloud_anomaly.evaluation import compare_detectors

comparison = compare_detectors(detections, labels_df)
comparison.round(3)

In [ ]:
from cloud_anomaly.benchmark import run as run_benchmark
from cloud_anomaly.evaluation import bootstrap_f1_ci, paired_significance

result = run_benchmark(n_seeds=10, n_days=90, base_seed=2000)
ci = bootstrap_f1_ci(result.raw, 'stl', 'OVERALL', n_resamples=500)
sig = paired_significance(result.raw, 'stl', 'iforest', anomaly_type='OVERALL')
print(f'STL OVERALL F1: {ci["mean"]:.3f} (95% CI [{ci["lo"]:.3f}, {ci["hi"]:.3f}])')
print(f'STL vs Isolation Forest — paired Wilcoxon p-value: {sig["p_value"]:.2e}')

## 5. Forecast and monthly projection

In [ ]:
from cloud_anomaly.forecast import forecast_per_service, projected_monthly_spend

fcast = forecast_per_service(long, horizon=14)
proj = projected_monthly_spend(fcast)
proj

## 6. Alert clustering — turn rows into incidents

In [ ]:
from cloud_anomaly.clustering import cluster_alerts, summarize_incidents

clustered = cluster_alerts(alerts_stl, eps=0.85, min_samples=2)
incidents = summarize_incidents(clustered)
incidents

## 7. Time-to-detect + cost-saved estimate

In [ ]:
from cloud_anomaly.evaluation import time_to_detect, cost_saved_estimate

ttd = time_to_detect(detections['stl'], labels_df)
saved = cost_saved_estimate(cur_df, detections['stl'], labels_df)
print(f'Median time-to-detect (STL): {ttd["days_to_detect"].median():.1f} days')
print(f'$ savable (STL, 1-day lag): ${saved["saved"]:,.0f} ({saved["ratio"]*100:.0f}% of leak)')

## 8. Real AWS CUR sanity check

The same pipeline can ingest a real AWS CUR file (`examples/aws_cur_sample.csv`) — just swap the data source. No detector code changes.

In [ ]:
from cloud_anomaly.cur_loader import load_cur_csv

real = load_cur_csv(ROOT / 'examples' / 'aws_cur_sample.csv')
real.head()